In [ ]:
"""分析美妆问卷中三、四线及以下城市受访者行为。

把本脚本与“3CE问卷数据-2026-08-12.csv”放在同一文件夹，直接运行即可。
依赖：pip install pandas openpyxl matplotlib
"""

In [ ]:
from __future__ import annotations

In [ ]:
import argparse
import re
from collections import Counter
from pathlib import Path

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from openpyxl.styles import Font, PatternFill

In [ ]:
COLUMNS = {
    "id": "记录ID", "submit_time": "提交时间", "age": "年龄", "city": "所在城市级别",
    "platform": "接触美妆内容的平台", "reason": "购买彩妆的主要原因",
    "difficulty": "购买彩妆的困难", "mismatch": "种草后发现不适合自己的频率",
    "brand_need": "最希望美妆品牌解决的问题", "q9": "3CE了解程度",
    "q10a": "Q10A｜持续购买3CE的原因", "q11a": "Q11A｜提高购买频率的方式",
    "q10b": "Q10B｜很少或未继续购买的原因", "q11b": "Q11B｜重新关注或购买的方式",
    "q10c": "Q10C｜知道但未购买的原因", "q11c": "Q11C｜提高首次购买可能性的体验",
    # 按 CSV 实际列名读取（与原问卷题号含义疑似相反）
    "usual_brand": "Q10D｜通常使用的彩妆品牌",
    "new_brand_reason": "Q11D｜关注新彩妆品牌的原因",
    "kdrama": "韩系影视内容关注度", "kdrama_element": "影响兴趣的韩剧元素",
    "role_identification": "影视角色代入感", "style": "3CE韩剧女主妆风格",
    "brand_info": "希望品牌了解的信息", "value": "化妆的最大价值",
    "q16a": "体验评分：韩剧角色测试", "q16b": "体验评分：场景妆容助手",
    "q16c": "体验评分：AI个人风格探索", "q16d": "体验评分：3CE女性圈层社区",
    "q16e": "体验评分：韩系潮流实验室", "open_text": "Q18品牌长期陪伴期待",
}

In [ ]:
MULTI = ["platform", "reason", "difficulty", "q10a", "q11a", "q10b", "q11b",
         "q10c", "q11c", "new_brand_reason", "kdrama_element", "brand_info"]
SINGLE = ["age", "city", "mismatch", "brand_need", "q9", "kdrama",
          "role_identification", "style", "value"]
SCORES = ["q16a", "q16b", "q16c", "q16d", "q16e"]

In [ ]:
def read_csv(path: Path) -> pd.DataFrame:
    errors = []
    for enc in ("utf-8-sig", "utf-8", "gb18030"):
        try:
            return pd.read_csv(path, encoding=enc, dtype=str, keep_default_na=False)
        except (UnicodeDecodeError, pd.errors.ParserError) as exc:
            errors.append(f"{enc}: {exc}")
    raise RuntimeError("无法读取 CSV；已尝试 UTF-8/GB18030。\n" + "\n".join(errors))

In [ ]:
def clean_text(value: object) -> str:
    return re.sub(r"\s+", " ", str(value)).strip() if value is not None else ""

In [ ]:
def split_multi(value: object) -> list[str]:
    text = clean_text(value)
    if not text:
        return []
    # 不拆普通中文逗号，除非数据确实使用其作为多选分隔符；常见导出格式均覆盖。
    parts = re.split(r"\s*(?:\||;|；|、|，|,|\n|\r|/|／)\s*", text)
    return [p.strip(" []'\"") for p in parts if p.strip(" []'\"")]

In [ ]:
def lower_tier_mask(series: pd.Series) -> pd.Series:
    """纳入明确的三线、四线、五线、低线、县城和三线及以下；排除一/新一/二线。"""
    s = series.fillna("").astype(str).str.replace(r"\s+", "", regex=True)
    positive = s.str.contains(r"三线|四线|五线|六线|低线|县城|乡镇|农村|三线及以下", regex=True)
    negative = s.str.fullmatch(r"一线城市?|新一线城市?|二线城市?")
    return positive & ~negative

In [ ]:
def tier_group(value: object) -> str:
    s = clean_text(value)
    if "三线及以下" in s or any(x in s for x in ["四线", "五线", "六线", "低线", "县城", "乡镇", "农村"]):
        return "四线及以下/县城（含问卷合并项）"
    if "三线" in s:
        return "三线城市"
    return "其他"

In [ ]:
def frequency_table(df: pd.DataFrame, col: str, multi: bool = False) -> pd.DataFrame:
    base = len(df)
    if multi:
        counter = Counter(item for value in df[col] for item in split_multi(value))
        out = pd.DataFrame(counter.items(), columns=["选项", "选择人数"])
        out["占受访者比例"] = out["选择人数"] / base if base else 0
        return out.sort_values("选择人数", ascending=False, ignore_index=True)
    s = df[col].map(clean_text).replace("", "未回答")
    out = s.value_counts(dropna=False).rename_axis("选项").reset_index(name="人数")
    out["占比"] = out["人数"] / base if base else 0
    return out

In [ ]:
def branch_validity(df: pd.DataFrame) -> pd.DataFrame:
    rules = {
        "经常购买 → A": (r"经常购买", ["q10a", "q11a"]),
        "买过1-2次 → B": (r"买过|1\s*[-—至~]\s*2", ["q10b", "q11b"]),
        "听说过未购买 → C": (r"听说过|没有购买|未购买", ["q10c", "q11c"]),
        "完全不了解 → D": (r"完全不了解", ["usual_brand", "new_brand_reason"]),
    }
    rows = []
    q9 = df["q9"].map(clean_text)
    for label, (pattern, fields) in rules.items():
        subset = df[q9.str.contains(pattern, regex=True, na=False)]
        complete = subset[fields].apply(lambda x: x.map(clean_text).ne("").all(), axis=1).sum()
        rows.append([label, len(subset), int(complete), complete / len(subset) if len(subset) else 0])
    return pd.DataFrame(rows, columns=["分支", "应答人数", "两题均完成", "完成率"])

In [ ]:
def keyword_table(series: pd.Series) -> pd.DataFrame:
    themes = {
        "个性化推荐/懂我": r"个性|适合|懂我|推荐|色号|风格",
        "教程/专业指导": r"教程|教学|指导|教我|技巧|顾问",
        "场景妆容": r"场景|面试|约会|旅行|聚会|通勤|上班|校园",
        "试妆/试用": r"试妆|试用|小样|体验",
        "优惠/性价比": r"优惠|折扣|价格|性价比|会员|福利",
        "新品/潮流": r"新品|潮流|趋势|更新|限定|联名",
        "互动/社区/陪伴": r"互动|社区|陪伴|交流|分享|共创|活动",
        "品质/效果": r"品质|质量|持久|效果|安全|成分",
    }
    values = series.map(clean_text)
    answered = values.ne("").sum()
    rows = []
    for theme, pattern in themes.items():
        count = values.str.contains(pattern, regex=True, case=False, na=False).sum()
        rows.append([theme, int(count), count / answered if answered else 0])
    return pd.DataFrame(rows, columns=["主题", "提及人数", "占开放题有效回答比例"]).sort_values("提及人数", ascending=False)

In [ ]:
def main() -> None:
    parser = argparse.ArgumentParser(description="三四线及以下城市美妆问卷分析")
    script_dir = Path(__file__).resolve().parent
    parser.add_argument(
        "csv", type=Path, nargs="?",
        default=script_dir / "3CE问卷数据-2026-08-12.csv",
        help="原始 CSV 文件（不填写时读取脚本同目录下的默认文件）",
    )
    parser.add_argument(
        "-o", "--output", type=Path,
        default=script_dir / "3CE问卷分析结果",
        help="结果文件夹",
    )
    args = parser.parse_args()
    if not args.csv.exists():
        raise FileNotFoundError(
            f"找不到 CSV 文件：{args.csv}\n"
            "请确认脚本和 3CE问卷数据-2026-08-12.csv 位于同一文件夹。"
        )
    args.output.mkdir(parents=True, exist_ok=True)

    raw = read_csv(args.csv)
    missing = [v for v in COLUMNS.values() if v not in raw.columns]
    if missing:
        raise KeyError("CSV 缺少以下字段：\n- " + "\n- ".join(missing))
    df = raw.rename(columns={v: k for k, v in COLUMNS.items()}).copy()
    target = df[lower_tier_mask(df["city"])].copy()
    target["城市层级分析组"] = target["city"].map(tier_group)

    summary = pd.DataFrame([
        ["全部有效记录", len(df)], ["三四线及以下记录", len(target)],
        ["目标样本占比", len(target) / len(df) if len(df) else 0],
        ["Q18有效文本数", target["open_text"].map(clean_text).ne("").sum()],
    ], columns=["指标", "值"])

    score_long = target.melt(id_vars=["城市层级分析组"], value_vars=SCORES,
                             var_name="体验", value_name="评分")
    score_long["体验"] = score_long["体验"].map({k: COLUMNS[k] for k in SCORES})
    score_long["评分"] = pd.to_numeric(score_long["评分"].str.extract(r"([1-5])")[0], errors="coerce")
    score_summary = score_long.groupby("体验")["评分"].agg(["count", "mean", "median", "std"]).reset_index()
    score_summary.columns = ["体验", "有效样本", "均分", "中位数", "标准差"]
    score_by_tier = score_long.pivot_table(index="体验", columns="城市层级分析组", values="评分", aggfunc="mean").reset_index()

    sheets: dict[str, pd.DataFrame] = {
        "样本概览": summary, "Q9分支完成度": branch_validity(target),
        "Q16体验评分": score_summary, "Q16分城市层级": score_by_tier,
        "Q18主题词": keyword_table(target["open_text"]), "Q18原文": target[["id", "city", "open_text"]],
        "目标样本明细": target,
    }
    for key in SINGLE:
        sheets[f"单选_{key}"] = frequency_table(target, key)
    for key in MULTI:
        sheets[f"多选_{key}"] = frequency_table(target, key, multi=True)

    # 城市层级交叉表：对关键行为题展示人数及行百分比。
    for key in ["age", "mismatch", "brand_need", "q9", "kdrama", "role_identification", "value"]:
        ct = pd.crosstab(target["城市层级分析组"], target[key].map(clean_text).replace("", "未回答"))
        pct = pd.crosstab(target["城市层级分析组"], target[key].map(clean_text).replace("", "未回答"), normalize="index")
        cross = pd.concat({"人数": ct, "行百分比": pct}, axis=1).reset_index()
        # pandas 暂不支持在 index=False 时写入 MultiIndex 列，因此先压平成普通列名。
        cross.columns = [
            "城市层级分析组" if str(a) == "城市层级分析组" else f"{a}｜{b}"
            for a, b in cross.columns
        ]
        sheets[f"交叉_{key}"] = cross

    xlsx = args.output / "三四线美妆问卷分析.xlsx"
    with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
        for name, table in sheets.items():
            safe = name[:31]
            table.to_excel(writer, sheet_name=safe, index=False)
            ws = writer.book[safe]
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
            for cell in ws[1]:
                cell.font = Font(name=cell.font.name or "微软雅黑", size=cell.font.sz or 11,
                                 bold=True, color="FFFFFF")
                cell.fill = PatternFill("solid", fgColor="8F315B")
            for col_cells in ws.columns:
                width = min(max(len(str(c.value or "")) for c in list(col_cells)[:200]) + 2, 45)
                ws.column_dimensions[col_cells[0].column_letter].width = max(width, 10)
            for row in ws.iter_rows():
                for cell in row:
                    if isinstance(cell.value, float) and ("比例" in str(ws.cell(1, cell.column).value) or "占比" in str(ws.cell(1, cell.column).value) or "完成率" in str(ws.cell(1, cell.column).value)):
                        cell.number_format = "0.0%"

    # 便于其他分析工具继续使用的目标样本 CSV。
    target.rename(columns={k: v for k, v in COLUMNS.items() if k in target.columns}).to_csv(
        args.output / "三四线及以下目标样本.csv", index=False, encoding="utf-8-sig")

    plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
    plt.rcParams["axes.unicode_minus"] = False
    chart = score_summary.sort_values("均分")
    ax = chart.plot.barh(x="体验", y="均分", legend=False, color="#B44575", figsize=(10, 5))
    ax.set_xlim(0, 5); ax.set_xlabel("平均兴趣评分（1-5）"); ax.set_ylabel("")
    ax.set_title("三四线及以下受访者：3CE体验兴趣评分")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", padding=3)
    plt.tight_layout(); plt.savefig(args.output / "Q16体验评分.png", dpi=180); plt.close()
    print("\n========== 三四线及以下人群核心分析 ==========")
    print(f"目标样本：{len(target)} 人，占全部样本的 {len(target) / len(df):.1%}")

    print("\n【3CE认知与购买情况】")
    print(frequency_table(target, "q9").to_string(index=False))

    print("\n【主要美妆内容平台】")
    print(frequency_table(target, "platform", multi=True).head(5).to_string(index=False))

    print("\n【购买彩妆的主要原因】")
    print(frequency_table(target, "reason", multi=True).head(5).to_string(index=False))

    print("\n【购买彩妆的主要困难】")
    print(frequency_table(target, "difficulty", multi=True).head(5).to_string(index=False))

    print("\n【最希望品牌解决的问题】")
    print(frequency_table(target, "brand_need").to_string(index=False))

    print("\n【Q16体验兴趣评分】")
    print(
        score_summary.sort_values("均分", ascending=False)
        .round({"均分": 2, "中位数": 2, "标准差": 2})
        .to_string(index=False)
    )

    print("\n【Q18长期陪伴期待主题】")
    print(keyword_table(target["open_text"]).head(8).to_string(index=False))

    print("\n============================================")
    print(f"完成：目标样本 {len(target)}/{len(df)} 人")
    print(f"Excel：{xlsx.resolve()}")

In [ ]:
if __name__ == "__main__":
    main()